# Bioinformatics pipeline (DADA2 and VSEARCH)

This notebook implements DADA2 and VSEARCH _de novo_ pipelines for amplicon data, covering all steps through to taxonomy assignment.

The pipeline includes:
- Removal of Ns and primers (cutadapt)
- Read quality control, filtering, and trimming (filterAndTrim)
- Error modelling, dereplication, denoising, read merging and chimera removal (DADA2 & VSEARCH)

The DADA2 workflow is adapted from the [NEOF version](https://github.com/khmaher/HPC_dada2/tree/main?tab=readme-ov-file), and the VSEARCH workflow is based on a published [GitHub pipeline](https://github.com/torognes/vsearch) and [VSEARCH documentation](https://torognes.github.io/vsearch/), both modified for this project.

Taxonomic assignment is performed using three options:

- RDP (assignTaxonomy), BLAST, and MapSeq
- A custom reference database combining MetaFishLib (fish) and MIDORI2 (non-fish)
- taxonomizr to summarise BLAST results

Currently using the standard Meta-Fish-Lib, with plans to incorporate a custom fish database when Jono completed.

## Set-up

Install packages, load required functions and get data.

In [0]:
source("Scripts/00_setup.R") #packages and functions

In [0]:
%sh

apt-get -q update

# cutadapt
apt-get -q install -y cutadapt
#pip install -q cutadapt
#cutadapt --version

# vsearch install
apt-get -q install -y vsearch

# BLAST+ installed
apt-get -q install -y ncbi-blast+

# MAPseq
cd /dbfs/tmp # temp directory
wget https://github.com/jfmrod/MAPseq/releases/download/v2.1.1/mapseq-2.1.1-linux.tar.gz # download
tar -xvzf mapseq-2.1.1-linux.tar.gz # extract

## check install
cd /dbfs/tmp/mapseq-2.1.1-linux
./mapseq

In [0]:
test_data_name <- "Windermere_2017" # change here to update test data (same name as folder)
test_data_loc <- file.path("Data", "Raw", test_data_name)
results_loc = paste("Results/", test_data_name)
print(head(list.files(test_data_loc)))

In [0]:
manifest <- make_manifest(test_data_loc)
write.csv(manifest, file.path("Data", "Temp", test_data_name, "manifest.csv"), row.names = FALSE)
head(manifest)

## Removing Ns

In [0]:
source("Scripts/01_remove_Ns.R")

## Identify and remove primers

In [0]:
# Riaz 2011

FWD <- "ACTGGGATTAGATACCCC"
REV <- "TAGAACAGGCTCCTCTAG"

In [0]:
# these would be options in actual pipeline

minimum <- 60 #Minimum read length cutoff. Recommend >0
copies <- 2 #Number of copies of a primer to be removed as sometimes dulication can occur. Recommended minimum is 2

In [0]:
source("Scripts/02_primer_removal.R")

## Generate quality plots

In [0]:
source("Scripts/03_raw_quality_plots.R")

In the plots below:
- Grey-scale heatmap shows the frequency of each quality score along the read lengths - looking for over 30 ish. 
- Green line is median quality score. 
- Orange line are quartiles. 
- Red line at the bottom represent the proportion of reads of that particular length. 

Our reads on average are approx 100bp long after primers removed in the first two samples, which is about right for these primers.

In [0]:
print(plotQualityProfile(fnFs.cut[1:2]))

In [0]:
print(plotQualityProfile(fnRs.cut[1:2]))

## Cleaning the data (filterAndTrim)

Parameters for filterAndTrim include:
- **maxN** - after truncation, sequences with more then X Ns will be disgarded
- **truncQ** - Truncate reads at the first instance of a quality score less then or equal to X
- **rm.phix** - Discard reads that match against the phiX genome
- **maxEE** - After truncation, reads with higher than X expected errors will be discarded
- **minLen** - Remove reads with length less than 60 (note these should gave already been removed by cutadapt)
- **multithread** - input files are filtered in parallel (logical)
- **truncLen** - controls where each read is cut (truncated) based on its length

In [0]:
truncLen=c(80,95) # where the quality dropped in previous plots (RingTrial = 80 & 95)
maxEE <-  c(2,2) #maxEE value 
truncQ <- 2 #truncQ value
minLen <- 50 #Impose a minimum length cutoff

# other options in NEOF pipeline are subset (only run a portion of the data) and marker (for better labelling when running multiple markers)

In [0]:
source("Scripts/04_filterAndTrim.R")

In [0]:
plotQualityProfile(fnFs.filtN[5:6])

In [0]:
plotQualityProfile(fnRs.filtN[1:2])

## Generate error model

In [0]:
source("Scripts/05_generate_error_model.R")

In the plots below:
- Error rates for each possible transition (e.g. A->C, A->G) are shown
- Red line = expected based on quality score
- Black line = estimate 
- Black dots = observed 

We want black lines and black dots to match. We also want to see a rough negative correlation. 

If it looks weird, can increase the number of bases the function is using (default is 100 million).

In [0]:
plotErrors(errF, nominalQ = TRUE)


In [0]:
plotErrors(errR, nominalQ = TRUE)

Common to see 'hook' between 30 - 40 with NovaSeq and MiniSeq data when visualising in DADA2. Couple of forums have discussion on this topic. Not to be of too much concern unless seeing any weird results.

## Deplication, merging and chimera removal

### DADA2

In [0]:
source("Scripts/06a_derep_DADA2.R")

### VSEARCH de novo

Documentation for VSEARCH [here](https://torognes.github.io/vsearch/). Found bash code for VSEARCH here which I've adapted - https://github.com/mgdesaix/metabarcoding-pipeline. 

In [0]:
%sh
bash Scripts/06b_derep_VSEARCH.sh # must change data name in script

## Sequence tracking

In [0]:
source("Scripts/07a_sequence_tracking.R")

Input and filtered columns will be the same for both pipelines, as they were done in the filter and trim step. They are not repeated below.

Let's look at just the VSEARCH steps post filterAndTrim. They are in a different order the DADA2 (merging happens first in VSEARCH) so please take note of the column names.

In [0]:
%sh
bash Scripts/07b_sequence_tracking.sh # must change data name in script

The number of sequences should look roughly similar between the two methods.

We now need to get the data in a similar format to the DADA2 outputs.

In [0]:
source("Scripts/07c_format_VSEARCH.R")

Check out data outputs.

In [0]:
# load dataset
ASV_tab_DADA2 <- read.table(
  file = file.path("Data", "Processed", test_data_name, "06_ASV_counts_DADA2.tsv"),
  header = TRUE,
  sep = "\t",
  row.names = 1,
  check.names = FALSE
)

# inspect
display(head(ASV_tab_DADA2))
dim(ASV_tab_DADA2)

In [0]:
ASV_tab_VSEARCH <- read.table(
  file = file.path("Data", "Processed", test_data_name, "06b_ASV_counts_VSEARCH.tsv"),
  header = TRUE,
  sep = "\t",
  row.names = 1,
  check.names = FALSE
)

# inspect
display(head(ASV_tab_VSEARCH))
dim(ASV_tab_VSEARCH)

In [0]:
ASV_seq_DADA2 <- readDNAStringSet(file.path("Data", "Processed", test_data_name, "06_ASV_seqs_DADA2.fasta"))
ASV_seq_DADA2

In [0]:
ASV_seq_VSEARCH <- readDNAStringSet(file.path("Data", "Processed", test_data_name, "06b_ASV_seqs_VSEARCH.fasta"))
ASV_seq_VSEARCH

## Assign taxonomy

### RDP

In [0]:
source("Scripts/08a_assign_taxonomy_RDP.R") 

In [0]:
source("Scripts/08c_assign_taxonomy_RDP_VSEARCH.R")

### BLAST

Tidy MetaFishLib for BLAST.

In [0]:
%python
import pandas as pd
# Load your database
mfl_db = pd.read_csv("Data/Databases/Meta-fish-lib/Riaz/references.12s.riaz.cleaned.v268.csv")

# Clean sequences + names
mfl_db = mfl_db.dropna(subset=["gbAccession", "nucleotides"])
mfl_db["seq"] = mfl_db["nucleotides"].str.upper().str.replace(r"[^ACGT]", "", regex=True)
mfl_db["species"] = mfl_db["sciNameValid"].str.replace(r"[^A-Za-z ]", "", regex=True)

# Write FASTA
with open("Data/Databases/Meta-fish-lib/metafish_riaz.fasta", "w") as f:
    for _, row in mfl_db.iterrows():
        header = f">{row['gbAccession']} {row['species']}"
        f.write(header + "\n")
        f.write(row["seq"] + "\n")

In [0]:
%sh
# to make BLAST formatted database from curated Meta Fish Lb
makeblastdb -in /Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/Meta-fish-lib/metafish_riaz.fasta -dbtype nucl -out Data/Databases/12S_riaz_fish_db/12S_riaz_fish_db

Now run blast.

In [0]:
%sh
bash Scripts/08b_assign_taxonomy_BLAST.sh # must change data name in script, need to parse later

In [0]:
%sh
bash Scripts/08d_assign_taxonomy_BLAST_VSEARCH.sh # must change data name in script, need to parse later

#### Condense BLAST taxonomy

Condense taxonomy (lowest common ancestor) for BLAST outputs. 

In [0]:
min_pident <- 95
min_length <- 5
top_frac <- 0.95

In [0]:
source("Scripts/09_condensing_taxonomy.R")

### MAPSeq

In [0]:
%python
import csv
from collections import defaultdict

# ---- file paths ----
input_csv = "/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/Meta-fish-lib/Riaz/references.12s.riaz.cleaned.v268.csv"

base_dir = "/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/Meta-fish-lib/"

fasta_out = base_dir + "meta_fish_riaz_mapseq.fasta"
tax_out = base_dir + "meta_fish_riaz_mapseq.tax"
cluster_out = base_dir + "meta_fish_riaz_mapseq.fasta.mscluster"

# ---- storage ----
records = []
species_to_ids = defaultdict(list)

def clean(x):
    if x is None:
        return "unknown"
    x = x.strip()
    return x.replace(" ", "_") if x and x != "NA" else "unknown"

# ---- load csv ----
with open(input_csv, newline='', encoding="utf-8-sig") as csvfile:
    reader = csv.DictReader(csvfile)

    for row in reader:
        seq_id = row["gbAccession"].strip()
        sequence = row["nucleotides"].strip().upper()

        kingdom = clean(row.get("kingdom"))
        phylum = clean(row.get("phylum"))
        class_ = clean(row.get("class"))
        order = clean(row.get("order"))
        family = clean(row.get("family"))
        genus = clean(row.get("genus"))
        species = clean(row.get("sciNameValid"))

        # ensure species format
        if "_" not in species:
            species = f"{genus}_sp"

        records.append((seq_id, sequence, kingdom, phylum, class_, order, family, genus, species))
        species_to_ids[species].append(seq_id)

print(f"Loaded {len(records)} records")

# ---- write fasta ----
with open(fasta_out, "w") as f:
    for r in records:
        seq_id, sequence = r[0], r[1]
        f.write(f">{seq_id}\n{sequence}\n")

print("FASTA complete")

# ---- write taxa ----
with open(tax_out, "w") as f:

    # header          Kingdom   Phylum    Class     Order     Family    Genus     Species
    f.write("#cutoff: 0.00:0.08 0.70:0.35 0.70:0.35 0.70:0.35 0.80:0.25 0.92:0.08 0.95:0.05\n") # took from github values
    f.write("#name: MetaFish\n")
    f.write("#levels: Kingdom Phylum Class Order Family Genus Species\n")

    for r in records:
        seq_id, _, kingdom, phylum, class_, order, family, genus, species = r

        taxonomy = f"{kingdom};{phylum};{class_};{order};{family};{genus};{species}"

        f.write(f"{seq_id}\t{taxonomy}\n")

print("TAX file complete")

# ----write mcluster ----
#with open(cluster_out, "w") as f:
    #for i, (species, ids) in enumerate(species_to_ids.items(), start=1):
        #f.write(f"cluster{i}\t" + " ".join(ids) + "\n")

#print("MSCLUSTER complete")

In [0]:

%sh
# dataset name 
test_data_name="Windermere_2017" # remeber to change

# input ASVs
input_fasta_dada2="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/06_ASV_seqs_DADA2.fasta"

input_fasta_vsearch="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/06b_ASV_seqs_VSEARCH.fasta"

# MAPseq database 
db_dir="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/Meta-fish-lib"

db_fasta="${db_dir}/meta_fish_riaz_mapseq.fasta"
db_tax="${db_dir}/meta_fish_riaz_mapseq.tax"

# output 
out_file_dada2="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/mapseq_output_dada2.mseq"

out_file_vsearch="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/mapseq_output_vsearch.mseq"

# dada2
/dbfs/tmp/mapseq-2.1.1-linux/mapseq -nthreads 8 "$input_fasta_dada2" "$db_fasta" "$db_tax" > "$out_file_dada2"

# vsearch
/dbfs/tmp/mapseq-2.1.1-linux/mapseq -nthreads 8 "$input_fasta_vsearch" "$db_fasta" "$db_tax" > "$out_file_vsearch"

## Jono pipeline

In [0]:
%skip
%python
%run Scripts/00_setup.py #packages and functions

In [0]:
%skip
%sh
pip install /Workspace/Shared/monitoring/EA_diatom_dada2_pipeline_DASH/packages/dokdo

In [0]:
%skip
%python
test_data_loc = "Data/Raw/RingTrial_Sean/"
results_loc = "Results"
os.listdir(test_data_loc)[:5]

In [0]:
%skip
%python
file_paths = get_files(results_loc)
samples = sort_paths(file_paths, results_loc)
export(samples, results_loc)

In [0]:
%skip
%sh
#define the analysis location (not retained from other cell)
results_loc = "/Results/"

qiime tools import \
    --type 'SampleData[PairedEndSequencesWithQuality]' \
    --input-path $results_loc/pe-33-manifest \
    --output-path /tmp/paired-end-demux.qza \
    --input-format PairedEndFastqManifestPhred33V2 && rm $results_loc/*fastq.gz

qiime demux summarize \
    --i-data /tmp/paired-end-demux.qza \
    --o-visualization /tmp/paired-end-demux.qzv

mv /tmp/paired-end-demux.qza $results_loc/paired-end-demux.qza
mv /tmp/paired-end-demux.qzv $results_loc/paired-end-demux.qzv